# Phần 1 — Giải 10 câu Trắc nghiệm (MCQ)

**Mục tiêu**: Lụm 20/20 điểm Phần 1.

**Nguyên tắc**:
- Làm trên data **THÔ** (raw), chỉ filter/dropna theo yêu cầu của từng câu.
- KHÔNG clean toàn cục trước khi làm MCQ (clean toàn cục để dành cho Phần 2 & 3).
- Mỗi câu in ra: **logic giải** + **đáp án cuối cùng** + **đáp án trắc nghiệm A/B/C/D**.
- Mỗi câu đẫ thử **2 cách** để cross-check (Dùng SQL trên branch DE, và Python trên branch main).

## Cell 0 — Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# Đường dẫn data raw — sửa lại nếu cấu trúc folder khác
RAW_DIR = Path('../data/raw')

# Load 15 file CSV (chỉ load các bảng cần dùng cho MCQ)
products    = pd.read_csv(RAW_DIR / 'products.csv')
customers   = pd.read_csv(RAW_DIR / 'customers.csv')
geography   = pd.read_csv(RAW_DIR / 'geography.csv')
orders      = pd.read_csv(RAW_DIR / 'orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv(RAW_DIR / 'order_items.csv')
payments    = pd.read_csv(RAW_DIR / 'payments.csv')
returns     = pd.read_csv(RAW_DIR / 'returns.csv', parse_dates=['return_date'])
web_traffic = pd.read_csv(RAW_DIR / 'web_traffic.csv', parse_dates=['date'])
sales_train = pd.read_csv(RAW_DIR / 'sales.csv', parse_dates=['Date'])

print('Successfully loaded 9 tables for MCQ')
for name, df in [('products', products), ('customers', customers), ('geography', geography),
                  ('orders', orders), ('order_items', order_items), ('payments', payments),
                  ('returns', returns), ('web_traffic', web_traffic), ('sales_train', sales_train)]:
    print(f'  {name:15s} shape={df.shape}')

C:\Users\admin\AppData\Local\Temp\ipykernel_10852\759267290.py:16: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv(RAW_DIR / 'order_items.csv')


Successfully loaded 9 tables for MCQ
  products        shape=(2412, 8)
  customers       shape=(121930, 7)
  geography       shape=(39948, 4)
  orders          shape=(646945, 8)
  order_items     shape=(714669, 7)
  payments        shape=(646945, 4)
  returns         shape=(39939, 7)
  web_traffic     shape=(3652, 7)
  sales_train     shape=(3833, 3)


## Cell 1 — Audit nhẹ (Data Quality Check)

Chỉ KIỂM TRA, không sửa data. Mục đích: phát hiện sớm các bẫy.
- Key uniqueness (`order_id` trong `orders` phải unique).
- Cardinality `orders ↔ payments` có thật sự 1:1?
- Missing % các cột quan trọng (`age_group`, `gender`, `promo_id`).
- Range giá trị bất thường (cogs > price?).

In [ ]:
print('=== KEY UNIQUENESS ===')
print(f'orders.order_id unique?       {orders["order_id"].is_unique}')
print(f'products.product_id unique?   {products["product_id"].is_unique}')
print(f'customers.customer_id unique? {customers["customer_id"].is_unique}')
print(f'geography.zip unique?         {geography["zip"].is_unique}')

print('\n=== CARDINALITY orders ↔ payments ===')
n_orders = orders['order_id'].nunique()
n_pay_orders = payments['order_id'].nunique()
n_pay_rows = len(payments)
print(f'orders distinct order_id:    {n_orders}')
print(f'payments distinct order_id:  {n_pay_orders}')
print(f'payments rows:               {n_pay_rows}')
print(f'1:1 strict? {n_pay_orders == n_pay_rows == n_orders}')

print('\n=== MISSING % (cột quan trọng) ===')
print(f'customers.age_group     null %: {customers["age_group"].isna().mean()*100:.1f}%')
print(f'customers.gender        null %: {customers["gender"].isna().mean()*100:.1f}%')
print(f'order_items.promo_id    null %: {order_items["promo_id"].isna().mean()*100:.1f}%')
print(f'order_items.promo_id_2  null %: {order_items["promo_id_2"].isna().mean()*100:.1f}%')

print('\n=== SANITY: products có row nào cogs >= price không? ===')
n_bad = (products['cogs'] >= products['price']).sum()
print(f'Số row cogs >= price: {n_bad} / {len(products)}')

print('\n=== ORDER STATUS distribution ===')
print(orders['order_status'].value_counts(dropna=False))

print('\n=== Date ranges ===')
print(f'orders:      {orders["order_date"].min()} → {orders["order_date"].max()}')
print(f'sales_train: {sales_train["Date"].min()} → {sales_train["Date"].max()}')
print(f'web_traffic: {web_traffic["date"].min()} → {web_traffic["date"].max()}')

=== KEY UNIQUENESS ===
orders.order_id unique?       True
products.product_id unique?   True
customers.customer_id unique? True
geography.zip unique?         True

=== CARDINALITY orders ↔ payments ===
orders distinct order_id:    646945
payments distinct order_id:  646945
payments rows:               646945
1:1 strict? True

=== MISSING % (cột quan trọng) ===
customers.age_group     null %: 0.0%
customers.gender        null %: 0.0%
order_items.promo_id    null %: 61.3%
order_items.promo_id_2  null %: 100.0%

=== SANITY: products có row nào cogs >= price không? ===
Số row cogs >= price: 0 / 2412

=== ORDER STATUS distribution ===
order_status
delivered    516716
cancelled     59462
returned      36142
shipped       13773
paid          13577
created        7275
Name: count, dtype: int64

=== Date ranges ===
orders:      2012-07-04 00:00:00 → 2022-12-31 00:00:00
sales_train: 2012-07-04 00:00:00 → 2022-12-31 00:00:00
web_traffic: 2013-01-01 00:00:00 → 2022-12-31 00:00:00


---
## Q1 — Trung vị số ngày giữa hai lần mua liên tiếp

**Đề**: Khách hàng có >1 đơn → tính median số ngày giữa 2 đơn liên tiếp.

**Bẫy**:
- PHẢI `groupby('customer_id')` trước khi `diff()` để không tính nhầm sang khách khác.
- Đề KHÔNG nói loại đơn `cancelled` → giữ nguyên all status.

**Đáp án**: A) 30  B) 90  C) 180  D) 365

In [ ]:
# Cách 1: dùng groupby + diff (chuẩn)
o = orders[['customer_id', 'order_date']].copy()
o = o.sort_values(['customer_id', 'order_date'])

# Lọc khách có ≥ 2 đơn
n_orders_per_cust = o.groupby('customer_id').size()
multi_cust = n_orders_per_cust[n_orders_per_cust >= 2].index
o_multi = o[o['customer_id'].isin(multi_cust)].copy()

# diff() trong từng customer
o_multi['gap_days'] = o_multi.groupby('customer_id')['order_date'].diff().dt.days
median_gap = o_multi['gap_days'].median()
print(f'Median inter-order gap: {median_gap} days')
print(f'Mean: {o_multi["gap_days"].mean():.1f}, Q25: {o_multi["gap_days"].quantile(0.25)}, Q75: {o_multi["gap_days"].quantile(0.75)}')

# Cross-check với 4 lựa chọn
options = {'A': 30, 'B': 90, 'C': 180, 'D': 365}
best = min(options.items(), key=lambda kv: abs(kv[1] - median_gap))
print(f'\n>>> Đáp án Q1: {best[0]}) {best[1]} ngày  (median thực tế = {median_gap})')

Median inter-order gap: 144.0 days
Mean: 285.6, Q25: 46.0, Q75: 357.0

>>> Đáp án Q1: C) 180 ngày  (median thực tế = 144.0)


---
## Q2 — Segment có gross margin trung bình cao nhất

**Công thức (đề cho)**: margin = (price - cogs) / price

**Đáp án**: A) Premium  B) Performance  C) Activewear  D) Standard

In [ ]:
p = products.copy()
p['margin'] = (p['price'] - p['cogs']) / p['price']

margin_by_segment = p.groupby('segment')['margin'].mean().sort_values(ascending=False)
print('Gross margin trung bình theo segment:')
print(margin_by_segment.round(4))

best_segment = margin_by_segment.idxmax()
print(f'\n>>> Đáp án Q2: segment cao nhất = {best_segment!r}')
print('>>> Đáp án là D)')

Gross margin trung bình theo segment:
segment
Standard       0.3134
Premium        0.2854
All-weather    0.2842
Activewear     0.2656
Performance    0.2636
Balanced       0.2580
Trendy         0.2408
Everyday       0.2363
Name: margin, dtype: float64

>>> Đáp án Q2: segment cao nhất = 'Standard'
>>> Đáp án là D)


---
## Q3 — Lý do trả hàng phổ biến nhất với sản phẩm Streetwear

**Lưu ý**: Chỉ cần join 2 bảng `returns ↔ products` (KHÔNG cần `order_items`).

**Đáp án**: A) defective  B) wrong_size  C) changed_mind  D) not_as_described

In [ ]:
# Join returns với products theo product_id
df = returns.merge(products[['product_id', 'category']], on='product_id', how='left')

# Lọc Streetwear
street = df[df['category'] == 'Streetwear']
print(f'Số bản ghi return của Streetwear: {len(street)}')

reason_counts = street['return_reason'].value_counts(dropna=False)
print('\nPhân bố lý do trả hàng:')
print(reason_counts)

top_reason = reason_counts.idxmax()
print(f'\n>>> Đáp án Q3: {top_reason!r}')

Số bản ghi return của Streetwear: 21799

Phân bố lý do trả hàng:
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

>>> Đáp án Q3: 'wrong_size'


---
## Q4 — Traffic source có bounce_rate trung bình thấp nhất

**Đáp án**: A) organic_search  B) paid_search  C) email_campaign  D) social_media

In [ ]:
br = web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values()
print('Bounce rate TB theo traffic source (thấp -> cao):')
print(br.round(8))

lowest = br.idxmin()
print(f'\n>>> Đáp án Q4: {lowest!r}')

Bounce rate TB theo traffic source (thấp -> cao):
traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64

>>> Đáp án Q4: 'email_campaign'


---
## Q5 — % dòng order_items có promo (`promo_id` không null)

**Bẫy**: ĐỀ chỉ hỏi `promo_id`, KHÔNG hỏi `promo_id_2`. Đừng OR 2 cột.

**Đáp án**: A) 12%  B) 25%  C) 39%  D) 54%

In [ ]:
# Tính trung bình của true -> tỷ lệ rows có promo_id không null (vì True=1, False=0) 
pct = order_items['promo_id'].notna().mean() * 100 
print(f'% rows có promo_id không null: {pct:.2f}%')

# Cross-check: nếu tính cả promo_id_2 thì khác bao nhiêu?
pct_either = (order_items['promo_id'].notna() | order_items['promo_id_2'].notna()).mean() * 100
print(f'(Cross-check) % có promo_id HOẶC promo_id_2: {pct_either:.2f}%')

options = {'A': 12, 'B': 25, 'C': 39, 'D': 54}
best = min(options.items(), key=lambda kv: abs(kv[1] - pct))
print(f'\n>>> Đáp án Q5: {best[0]}) {best[1]}%  (giá trị thực = {pct:.2f}%)')

% rows có promo_id không null: 38.66%
(Cross-check) % có promo_id HOẶC promo_id_2: 38.66%

>>> Đáp án Q5: C) 39%  (giá trị thực = 38.66%)


---
## Q6 — Age group có số đơn TB/khách cao nhất

**Công thức (đề cho)**: tổng số đơn / số khách hàng trong nhóm.

**Bẫy**: Mẫu số = TẤT CẢ khách trong age_group đó (kể cả khách 0 đơn).

**Đáp án**: A) 55+  B) 25-34  C) 35-44  D) 45-54

In [ ]:
# Bước 1: chỉ giữ khách có age_group != null
cust_clean = customers.dropna(subset=['age_group']).copy()

# Bước 2: đếm số đơn của mỗi customer (chỉ trong tập khách hợp lệ)
n_orders_per_cust = orders.groupby('customer_id').size().rename('n_orders')
cust_clean = cust_clean.merge(
    n_orders_per_cust, 
    left_on='customer_id', 
    right_index=True, 
    how='left')

cust_clean['n_orders'] = cust_clean['n_orders'].fillna(0)  # khách 0 đơn

# Bước 3: groupby age_group, lấy mean (bao gồm cả khách 0 đơn ở mẫu số)
avg_orders = cust_clean.groupby('age_group').agg(
    total_orders=('n_orders', 'sum'),
    n_customers=('customer_id', 'count'),
)
avg_orders['avg_orders_per_cust'] = avg_orders['total_orders'] / avg_orders['n_customers']
avg_orders = avg_orders.sort_values('avg_orders_per_cust', ascending=False)
print(avg_orders.round(3))

best = avg_orders['avg_orders_per_cust'].idxmax()
print(f'\n>>> Đáp án Q6: age_group cao nhất = {best!r}')

           total_orders  n_customers  avg_orders_per_cust
age_group                                                
55+             72760.0        13457                5.407
45-54          124138.0        23172                5.357
35-44          170368.0        31920                5.337
25-34          190622.0        36342                5.245
18-24           89057.0        17039                5.227

>>> Đáp án Q6: age_group cao nhất = '55+'


---
## Q7 — Region có doanh thu cao nhất trong sales_train

**Bẫy lớn**: `sales_train.csv` KHÔNG có thông tin region. Phải tính revenue từ `order_items` × `orders` × `geography`.

**Lọc theo train period**: order_date <= 2022-12-31.

**Đáp án**: A) West  B) Central  C) East  D) Xấp xỉ bằng nhau

In [ ]:
# Bước 1: tính line revenue cho mỗi dòng order_items
oi = order_items.copy()
oi['line_revenue'] = oi['quantity'] * oi['unit_price'] - oi['discount_amount'].fillna(0)

# Bước 2: aggregate về order_id
order_rev = oi.groupby('order_id', as_index=False)['line_revenue'].sum()

# Bước 3: join với orders để lấy zip + order_date
df = order_rev.merge(orders[['order_id', 'zip', 'order_date']], on='order_id', how='left')

# Bước 4: lọc theo train period
df = df[df['order_date'] <= '2022-12-31']

# Bước 5: join với geography để lấy region
df = df.merge(geography[['zip', 'region']], on='zip', how='left')

# Bước 6: tổng doanh thu theo region
rev_by_region = df.groupby('region')['line_revenue'].sum().sort_values(ascending=False)
print('Tổng revenue theo region (train period):')
print(rev_by_region.apply(lambda x: f'{x:,.0f}'))

# Check tỷ lệ chênh lệch
ratio = rev_by_region.max() / rev_by_region.min()
print(f'\nRatio max/min: {ratio:.2f}')
if ratio < 1.1:
    print('>>> Q7: 3 vùng XẤP XỈ BẰNG NHAU → đáp án D')
else:
    print(f'>>> Đáp án Q7: region cao nhất = {rev_by_region.idxmax()!r}')

Tổng revenue theo region (train period):
region
East       7,291,150,819
Central    4,719,491,268
West       3,670,227,178
Name: line_revenue, dtype: object

Ratio max/min: 1.99
>>> Đáp án Q7: region cao nhất = 'East'


---
## Q8 — Payment method phổ biến nhất trong đơn cancelled

**Đáp án**: A) credit_card  B) cod  C) paypal  D) bank_transfer

In [ ]:
cancelled = orders[orders['order_status'] == 'cancelled']
print(f'Số đơn cancelled: {len(cancelled)}')

pm_counts = cancelled['payment_method'].value_counts(dropna=False)
print('\nPhân bố payment_method trong đơn cancelled:')
print(pm_counts)

top_pm = pm_counts.idxmax()
print(f'\n>>> Đáp án Q8: {top_pm!r}')

Số đơn cancelled: 59462

Phân bố payment_method trong đơn cancelled:
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64

>>> Đáp án Q8: 'credit_card'


---
## Q9 — Size có tỷ lệ trả hàng cao nhất

**BẪY CỰC LỚN**: Đề định nghĩa rõ:
> tỷ lệ trả hàng = **số bản ghi trong returns** / **số dòng trong order_items**

→ Đây là **count rows**, KHÔNG PHẢI sum quantity!

**Đáp án**: A) S  B) M  C) L  D) XL

In [ ]:
# Join với products để lấy size cho cả 2 bảng
oi_size = order_items.merge(products[['product_id', 'size']], on='product_id', how='left')
ret_size = returns.merge(products[['product_id', 'size']], on='product_id', how='left')

# Chỉ giữ 4 size hợp lệ
valid_sizes = ['S', 'M', 'L', 'XL']
oi_size = oi_size[oi_size['size'].isin(valid_sizes)]
ret_size = ret_size[ret_size['size'].isin(valid_sizes)]

# === Cách 1 (THEO ĐỀ): count rows ===
n_oi = oi_size.groupby('size').size()
n_ret = ret_size.groupby('size').size()
ratio_rows = (n_ret / n_oi).sort_values(ascending=False)
print('Cách 1 (THEO ĐỀ - count rows):')
print((ratio_rows * 100).round(2).astype(str) + '%')

# === Cách 2 (cross-check): sum quantity ===
qty_oi = oi_size.groupby('size')['quantity'].sum()
qty_ret = ret_size.groupby('size')['return_quantity'].sum()
ratio_qty = (qty_ret / qty_oi).sort_values(ascending=False)
print('\nCách 2 (cross-check - sum quantity):')
print((ratio_qty * 100).round(2).astype(str) + '%')

answer_q9 = ratio_rows.idxmax()
print(f'\n>>> Đáp án Q9 (theo đề): size = {answer_q9!r}')
if ratio_rows.idxmax() != ratio_qty.idxmax():
    print(f'Cần Check Gấp: 2 cách ra khác nhau! Cách qty: {ratio_qty.idxmax()!r}. Nộp theo cách đề (count rows).')

Cách 1 (THEO ĐỀ - count rows):
size
S     5.65%
L     5.62%
M     5.57%
XL    5.52%
dtype: object

Cách 2 (cross-check - sum quantity):
size
S     3.46%
L     3.43%
M      3.4%
XL    3.36%
dtype: object

>>> Đáp án Q9 (theo đề): size = 'S'


---
## Q10 — Installments có payment_value TB cao nhất

**Đáp án**: A) 1 kỳ  B) 3 kỳ  C) 6 kỳ  D) 12 kỳ

In [ ]:
print('Phân bố installments trong payments:')
print(payments['installments'].value_counts().sort_index())

avg_pay = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)
print('\nAvg payment_value theo installments (cao -> thấp):')
print(avg_pay.round(2))

# Lọc chỉ 4 lựa chọn của đề
options = [1, 3, 6, 12]
avg_pay_subset = avg_pay[avg_pay.index.isin(options)]
print('\nChỉ trong 4 lựa chọn của đề:')
print(avg_pay_subset.round(2))

best_inst = avg_pay_subset.idxmax()
print(f'\n>>> Đáp án Q10: {best_inst} kỳ')

Phân bố installments trong payments:
installments
1     262866
2       1094
3     218949
6     109910
12     54126
Name: count, dtype: int64

Avg payment_value theo installments (cao -> thấp):
installments
6     24446.65
3     24399.64
12    24245.77
1     24113.27
2       708.47
Name: payment_value, dtype: float64

Chỉ trong 4 lựa chọn của đề:
installments
6     24446.65
3     24399.64
12    24245.77
1     24113.27
Name: payment_value, dtype: float64

>>> Đáp án Q10: 6 kỳ


---
## Cell 12 — TỔNG HỢP ĐÁP ÁN

In [ ]:
ANSWERS = {
    'Q1': 'C',  # 30 / 90 / 180 / 365
    'Q2': 'D',  # Premium / Performance / Activewear / Standard
    'Q3': 'B',  # defective / wrong_size / changed_mind / not_as_described
    'Q4': 'C',  # organic / paid / email / social
    'Q5': 'C',  # 12 / 25 / 39 / 54
    'Q6': 'A',  # 55+ / 25-34 / 35-44 / 45-54
    'Q7': 'C',  # West / Central / East / equal
    'Q8': 'A',  # credit_card / cod / paypal / bank_transfer
    'Q9': 'A',  # S / M / L / XL
    'Q10': 'C', # 1 / 3 / 6 / 12
}

print('=== ĐÁP ÁN CUỐI CÙNG ===')
for q, a in ANSWERS.items():
    print(f'  {q}: {a}')

n_filled = sum(1 for a in ANSWERS.values() if a != '?')
print(f'\nĐã điền: {n_filled}/10 câu')

=== ĐÁP ÁN CUỐI CÙNG ===
  Q1: C
  Q2: D
  Q3: B
  Q4: C
  Q5: C
  Q6: A
  Q7: C
  Q8: A
  Q9: A
  Q10: C

Đã điền: 10/10 câu
